# Lab 2 — Distances, Normalization and the Nearest-Mean Classifier
**ENGS 3302 Pattern Recognition · Week 2** · Notes: *Week 2 — Features, Distance and the Nearest-Mean Classifier*

| Part | What you do | How |
|---|---|---|
| **1. Demo** | Distances, the units problem, scalers, a messy table, nearest mean | All code is given. **Watch the lab video, run each cell, change values** |
| **2. Your turn** | Small tasks on the same ideas | Fill the `# TODO` cells. A check cell tells you if you are right |

This lab has **no grade**. The graded work this week is `Assignment_W02`.
In Colab: *File → Save a copy in Drive* first, then run the cells from top to bottom.

In [ ]:
# Run this cell first. It loads the libraries and a helper that checks your answers.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import make_pipeline

def check(label, got, expected, tol=1e-2):
    """Prints a green tick if `got` is close to `expected`, otherwise a red cross."""
    try:
        if isinstance(expected, str):
            ok = str(got) == expected
        else:
            ok = got is not None and np.allclose(np.asarray(got, dtype=float),
                                                 np.asarray(expected, dtype=float), atol=tol)
    except Exception:
        ok = False
    print(f"✅ {label}" if ok else f"❌ {label}: expected {expected}, got {got}")

# The ten fish from the Notes: lightness, width (cm). 0 = salmon, 1 = sea bass.
fish = np.array([[2.5, 16.0], [3.0, 19.3], [3.8, 16.5], [4.5, 18.0], [5.6, 17.0],
                 [4.2, 20.5], [5.2, 19.0], [6.0, 21.0], [6.8, 19.5], [7.5, 20.0]])
labels = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
names = np.array(["salmon", "sea bass"])

# Palmer Penguins (Horst, Hill & Gorman 2020, CC0): 344 penguins of 3 species, measured in mm and g.
# Uses penguins.csv from the Moodle zip if it is next to the notebook, otherwise downloads it.
PENGUIN_FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
def load_penguins():
    """Returns X (4 measurements), y (0 = Adelie, 1 = Chinstrap, 2 = Gentoo) and the cleaned table."""
    try:
        df = pd.read_csv("penguins.csv")
    except FileNotFoundError:
        df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
    df = df.dropna(subset=PENGUIN_FEATURES).reset_index(drop=True)   # 2 penguins have no measurements
    y = df["species"].map({"Adelie": 0, "Chinstrap": 1, "Gentoo": 2}).to_numpy()
    return df[PENGUIN_FEATURES].to_numpy(), y, df
print("Ready.")

---
## Part 1 — Demo *(follow the lab video)*
### 1.1 Two distances
Euclidean: square the differences, add, take the square root. Manhattan: add the absolute differences.

In [ ]:
a, b = fish[0], fish[5]          # a salmon and a sea bass
print("a =", a, " b =", b)
print("Euclidean:", np.linalg.norm(a - b))       # 4.81
print("Manhattan:", np.sum(np.abs(a - b)))       # 6.2

### 1.2 The nearest-mean classifier, by hand
One mean (centre) per class. A new fish goes to the class with the nearest centre.

In [ ]:
centres = np.array([fish[labels == c].mean(axis=0) for c in (0, 1)])
print("centres:\n", centres)                     # salmon (3.88, 17.36), sea bass (5.94, 20.0)

new = np.array([7.0, 18.0])
d = np.linalg.norm(centres - new, axis=1)        # distance to each centre
print("distances:", d.round(2), "->", names[d.argmin()])

### 1.3 When the units decide
Measure width in **millimetres** (multiply column 1 by 10) and ask again. The fish is the same; the answer is not.

In [ ]:
to_mm = np.array([1, 10])
fish_mm, new_mm = fish * to_mm, new * to_mm
centres_mm = np.array([fish_mm[labels == c].mean(axis=0) for c in (0, 1)])
d_mm = np.linalg.norm(centres_mm - new_mm, axis=1)
print("distances in mm:", d_mm.round(2), "->", names[d_mm.argmin()])

fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, F, C, q, unit in [(axs[0], fish, centres, new, "cm"), (axs[1], fish_mm, centres_mm, new_mm, "mm")]:
    ax.scatter(*F[labels == 0].T, c="tab:orange", label="salmon")
    ax.scatter(*F[labels == 1].T, c="tab:blue", marker="s", label="sea bass")
    ax.scatter(*C.T, c=["tab:orange", "tab:blue"], marker="X", s=200, edgecolor="k")
    ax.scatter(*q, c="white", edgecolor="k", marker="*", s=300)
    ax.set_xlabel("lightness"); ax.set_ylabel(f"width ({unit})")
axs[0].legend(); plt.tight_layout(); plt.show()

### 1.4 Normalization with scikit-learn
`fit` learns the numbers (mean and standard deviation, or min and max) **from the training data**.
`transform` applies the **same** numbers to any data, including new samples.

In [ ]:
z = StandardScaler().fit(fish)
print("means:", z.mean_, " std:", z.scale_.round(3))
print("new fish as z-scores:", z.transform([new]).round(3))     # [1.367, -0.415]

mm = MinMaxScaler().fit(fish)
print("new fish after min-max:", mm.transform([new]))          # [0.9, 0.4]

# z-scores do not care about the unit: same result in cm and in mm
z_mm = StandardScaler().fit(fish_mm)
print("same fish, in mm:", z_mm.transform([new_mm]).round(3))

### 1.5 A messy table: missing values and a categorical feature
Real tables have gaps and text columns. This small table is **made up** for practice.

In [ ]:
df = pd.DataFrame({
    "lightness": [2.5, 3.0, np.nan, 4.5, 6.0, 6.8],
    "width_cm":  [16.0, 19.3, 16.5, 18.0, 21.0, np.nan],
    "port":      ["Gaza", "Rafah", "Gaza", "Khan Younis", "Rafah", "Gaza"],
    "species":   ["salmon", "salmon", "salmon", "salmon", "sea bass", "sea bass"]})
print(df.isna().sum())                          # how many gaps per column

filled = df.fillna(df.mean(numeric_only=True))  # fill gaps with the column mean
coded = pd.get_dummies(filled, columns=["port"], dtype=int)   # one 0/1 column per port
coded

### 1.6 Nearest mean on real data: does normalization help?
We keep 30% of each data set aside as **test data**, train on the rest, and count correct answers on the test data.

In [ ]:
for name, loader in [("Penguins", lambda: load_penguins()[:2]), ("Iris", lambda: load_iris(return_X_y=True))]:
    X, y = loader()
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    raw = NearestCentroid().fit(Xtr, ytr).score(Xte, yte)
    scaled = make_pipeline(StandardScaler(), NearestCentroid()).fit(Xtr, ytr).score(Xte, yte)
    print(f"{name}: raw {raw:.3f}   z-score {scaled:.3f}")

Penguins mixes millimetres (bill depth 13.1–21.5 mm) with grams (body mass 2700–6300 g), so without scaling the body mass decides almost everything.
Iris has four lengths in cm; there, scaling makes a noisy feature as loud as the useful ones.
**Normalize when units or ranges are unrelated to the classes, and always check on test data.**

---
## Part 2 — Your turn
### Task 1 — Distances
Compute the Euclidean and the Manhattan distance between fish 2 (`fish[2]`) and fish 7 (`fish[7]`).

In [ ]:
euclid = None     # TODO
manhattan = None  # TODO

In [ ]:
check("Euclidean distance", euclid, 5.009)
check("Manhattan distance", manhattan, 6.7)

### Task 2 — z-scores by hand
Compute the z-scores of a new fish `[5.0, 19.0]`, using the mean and standard deviation of `fish` (`np.mean`, `np.std` with `axis=0`).

In [ ]:
z_new = None  # TODO: a NumPy array with two numbers

In [ ]:
check("z-scores of [5.0, 19.0]", z_new, [0.059, 0.195])

### Task 3 — Your own nearest-mean classifier
Complete the two functions. `fit_means` returns one centre per class (rows in class order 0, 1, ...).
`predict` returns the class of the nearest centre for every row of `X`.

In [ ]:
def fit_means(X, y):
    # TODO: return an array with one row (the mean) per class
    return None

def predict(means, X):
    # TODO: for each row of X, return the index of the nearest mean
    return None

In [ ]:
m = fit_means(fish, labels)
done = m is not None and predict(m, fish) is not None
check("centres", m if done else None, [[3.88, 17.36], [5.94, 20.0]])
check("training errors", int(np.sum(predict(m, fish) != labels)) if done else None, 0)
check("new fish [7.0, 18.0]", int(predict(m, np.array([[7.0, 18.0]]))[0]) if done else None, 1)

### Task 4 — Normalization changes Penguins
Use the Penguins split below. Compute the test accuracy of **your** classifier from Task 3 (a) on raw features and (b) after `StandardScaler` fitted on `Xtr` only.

In [ ]:
X, y, _ = load_penguins()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
acc_raw = None     # TODO
acc_scaled = None  # TODO

In [ ]:
check("accuracy, raw", acc_raw, 0.631)
check("accuracy, scaled", acc_scaled, 0.971)

### Reflect (2–3 sentences)
In Task 4 the scaler was fitted on `Xtr` only. What would be wrong with fitting it on all of `X`?

*Your answer:*